In [2]:
import zipfile
from pathlib import Path
import numpy as np
import scipy.io
from scipy.signal import butter, filtfilt, iirnotch

In [3]:
# =========================
# CONFIG
# =========================
ZIP_ROOT = Path(r"D:\SWEC ETHZ dataset")    
EXTRACT_ROOT = ZIP_ROOT / "_extracted"           
OUT_ROOT = Path(r"D:\swec_derivatives")

EXTRACT_ROOT.mkdir(exist_ok=True)
OUT_ROOT.mkdir(exist_ok=True)

FS_DEFAULT = 512.0
PRE_POST_SEC = 3 * 60
NOTCH_FREQ = 50.0
Q = 30.0 # notch sharpness

WIN_SEC = 2.0
HOP_SEC = 1.0
AMP_UV_THRESH = 5000.0  

# If preprocessing changed to regenerate all outputs:
REPROCESS_EXISTING = False

In [4]:
# =========================
# Filtering helpers
# =========================
def notch_filter(x, fs, f0=50.0, q=30.0):
    b, a = iirnotch(w0=f0, Q=q, fs=fs)
    return filtfilt(b, a, x, axis=-1)

def common_average_reference(x):
    # x: (n_channels, n_samples)
    return x - np.mean(x, axis=0, keepdims=True)

In [5]:
# =========================
# Data loading
# =========================
def extract_zip(zip_path: Path):
    print(f"[UNZIP] {zip_path.name}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(EXTRACT_ROOT)

def extract_all_zips():
    # match ID1.zip, ID13a.zip, etc.
    zip_files = sorted(ZIP_ROOT.glob("ID*.zip"))
    if not zip_files:
        raise RuntimeError(f"No ID*.zip files found under {ZIP_ROOT}")

    for zip_path in zip_files:
        pid = zip_path.stem  # e.g., ID1, ID13a

        # we consider it extracted if we can find any Sz*.mat under extracted/pid
        already = list((EXTRACT_ROOT / pid).rglob("Sz*.mat")) + list((EXTRACT_ROOT / pid).rglob("sz*.mat"))
        if already:
            print(f"[SKIP unzip] {pid} (found {len(already)} sz mats)")
            continue

        extract_zip(zip_path)

In [6]:
# =========================
# Data loading
# =========================

def load_swec_mat(mat_path: Path):
    mat = scipy.io.loadmat(mat_path, squeeze_me=True, struct_as_record=False)
    if "EEG" not in mat:
        raise KeyError(f"{mat_path.name}: missing EEG")

    eeg = np.asarray(mat["EEG"])
    if eeg.ndim != 2:
        raise ValueError(f"{mat_path.name}: EEG has shape {eeg.shape}")

    # force (channels, samples)
    if eeg.shape[0] > eeg.shape[1]:
        eeg = eeg.T
    if eeg.shape[0] > eeg.shape[1]:
        eeg = eeg.T

    fs = FS_DEFAULT
    for k in ["fs", "Fs", "srate", "sampling_rate"]:
        if k in mat:
            try:
                fs = float(np.asarray(mat[k]).item())
                break
            except Exception:
                pass

    return eeg.astype(np.float32), fs

def infer_patient_id(mat_path: Path) -> str:
    # find a parent folder starting with ID (ID1, ID13a, ID14b, etc.)
    for p in mat_path.parents:
        name = p.name
        if name.upper().startswith("ID"):
            return name
    # fallback: use immediate parent
    return mat_path.parent.name



In [7]:
# =========================
# Windowing + labeling
# =========================
def make_windows(x, fs, start_idx, end_idx):
    win = int(WIN_SEC * fs)
    hop = int(HOP_SEC * fs)

    X, y, t0 = [], [], []
    n = x.shape[1]

    for s in range(0, n - win + 1, hop):
        seg = x[:, s:s+win]
        ictal = not (s + win <= start_idx or s >= end_idx)
        X.append(seg)
        y.append(1 if ictal else 0)
        t0.append(s)

    return np.stack(X), np.array(y, dtype=np.int64), np.array(t0, dtype=np.int64)

In [8]:
def process_all():
    extract_all_zips()

    # Find seizure mats anywhere under extracted
    mat_files = [p for p in EXTRACT_ROOT.rglob("*.mat") if "sz" in p.stem.lower()]
    if not mat_files:
        raise RuntimeError(f"No seizure .mat files found under {EXTRACT_ROOT}")

    print(f"\nFound {len(mat_files)} seizure files.")

    for mat_path in sorted(mat_files):
        patient_id = infer_patient_id(mat_path)
        seizure_id = mat_path.stem

        out_dir = OUT_ROOT / patient_id
        out_dir.mkdir(parents=True, exist_ok=True)
        out_file = out_dir / f"{patient_id}_{seizure_id}.npz"

        if out_file.exists() and not REPROCESS_EXISTING:
            print(f"[SKIP] {out_file.name}")
            continue

        print(f"\n[PROCESS] {patient_id} | {seizure_id} | {mat_path}")

        try:
            x, fs = load_swec_mat(mat_path)

            n_ch, n_samp = x.shape
            pre = int(PRE_POST_SEC * fs)
            start_idx = pre
            end_idx = n_samp - pre
            if end_idx <= start_idx:
                raise ValueError("Recording too short for 3min pre/post")

            # preprocess (unfiltered raw -> filtered)
            x = notch_filter(x, fs, NOTCH_FREQ, Q)
            x = common_average_reference(x)

            X, y, t0 = make_windows(x, fs, start_idx, end_idx)

            # artifact rejection
            max_amp = np.max(np.abs(X), axis=(1, 2))
            keep = max_amp < AMP_UV_THRESH
            X, y, t0 = X[keep], y[keep], t0[keep]

            np.savez_compressed(
                out_file,
                X=X, y=y, t0=t0,
                fs=np.array(fs, dtype=np.float32),
                start_idx=np.array(start_idx, dtype=np.int64),
                end_idx=np.array(end_idx, dtype=np.int64),
                patient_id=str(patient_id),
                seizure_id=str(seizure_id),
                source_mat=str(mat_path),
            )

            print(f"  Saved {out_file} | windows={len(y)} | ictal%={y.mean():.3f}")

        except Exception as e:
            print(f"  [ERROR] {patient_id}/{seizure_id}: {repr(e)}")

    print("\nDONE.")

process_all()

[SKIP unzip] ID1 (found 26 sz mats)
[SKIP unzip] ID10 (found 10 sz mats)
[SKIP unzip] ID11 (found 4 sz mats)
[SKIP unzip] ID12 (found 20 sz mats)
[SKIP unzip] ID13a (found 8 sz mats)
[SKIP unzip] ID13b (found 6 sz mats)
[SKIP unzip] ID14a (found 8 sz mats)
[SKIP unzip] ID14b (found 6 sz mats)
[SKIP unzip] ID15 (found 6 sz mats)
[SKIP unzip] ID16 (found 12 sz mats)
[SKIP unzip] ID2 (found 8 sz mats)
[SKIP unzip] ID3 (found 4 sz mats)
[SKIP unzip] ID4a (found 14 sz mats)
[SKIP unzip] ID4b (found 14 sz mats)
[SKIP unzip] ID5 (found 20 sz mats)
[SKIP unzip] ID6 (found 8 sz mats)
[SKIP unzip] ID7 (found 4 sz mats)
[SKIP unzip] ID8 (found 4 sz mats)
[SKIP unzip] ID9 (found 18 sz mats)

Found 100 seizure files.
[SKIP] ID1_Sz1.npz
[SKIP] ID1_Sz10.npz
[SKIP] ID1_Sz11.npz
[SKIP] ID1_Sz12.npz
[SKIP] ID1_Sz13.npz
[SKIP] ID1_Sz2.npz

[PROCESS] ID1 | Sz3 | D:\SWEC ETHZ dataset\_extracted\ID1\Sz3.mat
  Saved D:\swec_derivatives\ID1\ID1_Sz3.npz | windows=495 | ictal%=0.277
[SKIP] ID1_Sz4.npz
[SKIP] ID

In [9]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch

# =========================
# QC / PLOTTING CONFIG
# =========================
QC_PLOTS_DIR = OUT_ROOT / "_qc_plots"
QC_PLOTS_DIR.mkdir(exist_ok=True)

QC_MAX_PATIENTS = 6          # how many patients to plot
QC_FILES_PER_PATIENT = 1     # how many seizure files per patient
QC_SECONDS = 10              # duration of time segment to show
QC_N_CHANNELS = 4            # number of channels to plot (stacked)
QC_SEGMENT_POLICY = "middle" # "middle" or "ictal_middle" or "start"

def _choose_segment(n_samp, fs, start_idx=None, end_idx=None, seconds=10, policy="middle"):
    seg_len = int(seconds * fs)
    seg_len = min(seg_len, n_samp)
    if seg_len < 2:
        return 0, n_samp

    if policy == "start":
        s0 = 0
    elif policy == "ictal_middle" and start_idx is not None and end_idx is not None and end_idx > start_idx:
        mid = (start_idx + end_idx) // 2
        s0 = int(np.clip(mid - seg_len // 2, 0, n_samp - seg_len))
    else:
        # default: middle of the whole recording
        s0 = max(0, (n_samp - seg_len) // 2)

    return s0, s0 + seg_len

def _plot_before_after_one(mat_path, out_file, patient_id, seizure_id):
    # Load raw
    x_raw, fs = load_swec_mat(mat_path)   # (ch, samples)
    n_ch, n_samp = x_raw.shape

    # Compute seizure interval indices the same way as preprocessing (for optional ictal_middle segment)
    pre = int(PRE_POST_SEC * fs)
    start_idx = pre
    end_idx = n_samp - pre if n_samp > 2 * pre else None

    # Apply the SAME preprocessing, but ONLY in-memory for visualization (no saving)
    x_after = notch_filter(x_raw, fs, NOTCH_FREQ, Q)
    x_after = common_average_reference(x_after)

    # Pick channels (spread across montage)
    ch_sel = np.linspace(0, n_ch - 1, num=min(QC_N_CHANNELS, n_ch), dtype=int)

    # Choose a segment to plot
    s0, s1 = _choose_segment(
        n_samp, fs,
        start_idx=start_idx,
        end_idx=end_idx if end_idx is not None else None,
        seconds=QC_SECONDS,
        policy=QC_SEGMENT_POLICY
    )
    t = np.arange(s0, s1) / fs

    # ---- TIME-DOMAIN (stacked channels, normalized for readability) ----
    fig = plt.figure(figsize=(12, 7))
    ax1 = fig.add_subplot(2, 1, 1)

    offset = 0.0
    for i, ch in enumerate(ch_sel):
        raw = x_raw[ch, s0:s1]
        aft = x_after[ch, s0:s1]

        # normalize by robust amplitude (only for plotting)
        scale = np.percentile(np.abs(raw), 95) + 1e-6
        raw_n = raw / scale
        aft_n = aft / scale

        ax1.plot(t, raw_n + offset, lw=0.8, label="raw" if i == 0 else None)
        ax1.plot(t, aft_n + offset, lw=0.8, label="after notch+CAR" if i == 0 else None)
        offset += 3.0

    ax1.set_title(f"{patient_id} | {seizure_id} — Time domain (raw vs after)")
    ax1.set_xlabel("Time (s)")
    ax1.set_ylabel("Amplitude (normalized, stacked)")
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc="upper right")

    # ---- FREQUENCY-DOMAIN (Welch PSD on one channel) ----
    ax2 = fig.add_subplot(2, 1, 2)
    ch0 = int(ch_sel[0])

    raw_seg = x_raw[ch0, s0:s1]
    aft_seg = x_after[ch0, s0:s1]

    nperseg = min(len(raw_seg), int(2 * fs))  # up to 2-second Welch windows
    f1, p1 = welch(raw_seg, fs=fs, nperseg=nperseg)
    f2, p2 = welch(aft_seg, fs=fs, nperseg=nperseg)

    ax2.semilogy(f1, p1 + 1e-18, lw=1.0, label="raw")
    ax2.semilogy(f2, p2 + 1e-18, lw=1.0, label="after notch+CAR")

    ax2.set_xlim(0, min(150, fs / 2))  # show up to 150 Hz (adjust if you want)
    ax2.set_title(f"PSD (Welch) — channel {ch0} (check ~{NOTCH_FREQ:.0f} Hz notch)")
    ax2.set_xlabel("Frequency (Hz)")
    ax2.set_ylabel("Power")
    ax2.grid(True, alpha=0.3)
    ax2.legend(loc="upper right")

    # Save
    out_png = QC_PLOTS_DIR / patient_id / f"{patient_id}_{seizure_id}_before_after.png"
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(out_png, dpi=150)
    plt.close(fig)

    print(f"[QC SAVED] {out_png}")

def qc_generate_plots_only_for_preprocessed():
    """
    QC plots that respect the preprocessing skip logic:
    - It will ONLY generate plots for seizure mats where the corresponding .npz already exists.
    - If .npz is missing, it skips (so we don't 'reprocess' anything).
    """
    extract_all_zips()  # already has "skip unzip if extracted"

    # Find seizure mats
    mat_files = [p for p in EXTRACT_ROOT.rglob("*.mat") if "sz" in p.stem.lower()]
    if not mat_files:
        raise RuntimeError(f"No seizure .mat files found under {EXTRACT_ROOT}")
    print(f"Found {len(mat_files)} seizure .mat files.")

    # Group by patient
    by_pid = {}
    for p in sorted(mat_files):
        pid = infer_patient_id(p)
        by_pid.setdefault(pid, []).append(p)

    pids = sorted(by_pid.keys())[:QC_MAX_PATIENTS]
    print(f"QC plotting for up to {len(pids)} patients (max={QC_MAX_PATIENTS}).")

    for pid in pids:
        count = 0
        for mat_path in by_pid[pid]:
            seizure_id = mat_path.stem

            # Match EXACT output naming from preprocessing
            out_dir = OUT_ROOT / pid
            out_file = out_dir / f"{pid}_{seizure_id}.npz"

            # Respect skip logic: ONLY proceed if already preprocessed exists
            if not out_file.exists():
                continue

            try:
                print(f"[QC] {pid} | {seizure_id} (npz exists) -> plotting")
                _plot_before_after_one(mat_path, out_file, pid, seizure_id)
                count += 1
            except Exception as e:
                print(f"[QC ERROR] {pid}/{seizure_id}: {repr(e)}")

            if count >= QC_FILES_PER_PATIENT:
                break

        if count == 0:
            print(f"[QC SKIP] {pid}: no matching preprocessed .npz found (won't reprocess).")

    print("QC plotting done.")

# Run QC plotting (no npz generation)
qc_generate_plots_only_for_preprocessed()


[SKIP unzip] ID1 (found 26 sz mats)
[SKIP unzip] ID10 (found 10 sz mats)
[SKIP unzip] ID11 (found 4 sz mats)
[SKIP unzip] ID12 (found 20 sz mats)
[SKIP unzip] ID13a (found 8 sz mats)
[SKIP unzip] ID13b (found 6 sz mats)
[SKIP unzip] ID14a (found 8 sz mats)
[SKIP unzip] ID14b (found 6 sz mats)
[SKIP unzip] ID15 (found 6 sz mats)
[SKIP unzip] ID16 (found 12 sz mats)
[SKIP unzip] ID2 (found 8 sz mats)
[SKIP unzip] ID3 (found 4 sz mats)
[SKIP unzip] ID4a (found 14 sz mats)
[SKIP unzip] ID4b (found 14 sz mats)
[SKIP unzip] ID5 (found 20 sz mats)
[SKIP unzip] ID6 (found 8 sz mats)
[SKIP unzip] ID7 (found 4 sz mats)
[SKIP unzip] ID8 (found 4 sz mats)
[SKIP unzip] ID9 (found 18 sz mats)
Found 100 seizure .mat files.
QC plotting for up to 6 patients (max=6).
[QC] ID1 | Sz1 (npz exists) -> plotting
[QC SAVED] D:\swec_derivatives\_qc_plots\ID1\ID1_Sz1_before_after.png
[QC] ID10 | Sz1 (npz exists) -> plotting
[QC SAVED] D:\swec_derivatives\_qc_plots\ID10\ID10_Sz1_before_after.png
[QC] ID11 | Sz1 (